The demographic info file has 6 columns:
  - Patient number
  - Age
  - Sex
  - Adult BMI (kg/m2)
  - Child Weight (kg)
  - Child Height (cm)


Each audio file name is divided into 5 elements, separated with underscores (_).

1. Patient number (101,102,...,226)
2. Recording index
3. Chest location 
      a. Trachea (Tc)
      b. Anterior left (Al)
      c. Anterior right (Ar)
      d. Posterior left (Pl)
      e. Posterior right (Pr)
      f. Lateral left (Ll)
      g. Lateral right (Lr)
4. Acquisition mode 
     a. sequential/single channel (sc), 
     b. simultaneous/multichannel (mc)
5. Recording equipment 
     a. AKG C417L Microphone (AKGC417L), 
     b. 3M Littmann Classic II SE Stethoscope (LittC2SE), 
     c. 3M Litmmann 3200 Electronic Stethoscope (Litt3200), 
     d.  WelchAllyn Meditron Master Elite Electronic Stethoscope (Meditron)

The annotation text files have four columns:
- Beginning of respiratory cycle(s)
- End of respiratory cycle(s)
- Presence/absence of crackles (presence=1, absence=0)
- Presence/absence of wheezes (presence=1, absence=0)

The abbreviations used in the diagnosis file are:
- COPD: Chronic Obstructive Pulmonary Disease
- LRTI: Lower Respiratory Tract Infection
- URTI: Upper Respiratory Tract Infection

In [18]:
#check how many files I have in text and audio folder
import os

# path to the folder
audio_dir = '../raw_data/audio_and_txt_files/'

# List all files in the directory
all_files = os.listdir(audio_dir)

# Count extensions
wav_count = len([f for f in all_files if f.endswith('.wav')])
txt_count = len([f for f in all_files if f.endswith('.txt')])

print(f"--- inventory check ---")
print(f"total .wav files (Recordings): {wav_count}")
print(f"total .txt files (Annotations): {txt_count}")
print(f"total items in folder: {len(all_files)}")

# double chheck if every audio has a matching text file
if wav_count == txt_count:
    print("\n✅ good! every audio file has a matching txt file.")
else:
    print("\n⚠️ action needed: mismatch between audio and text")

--- inventory check ---
total .wav files (Recordings): 920
total .txt files (Annotations): 920
total items in folder: 1840

✅ good! every audio file has a matching txt file.


In [19]:
#math is not mathing

# id files that are NOT .wav or .txt
other_files = [f for f in all_files if not f.endswith('.wav') and not f.endswith('.txt')]

print(f"num of mysterious files: {len(other_files)}")
if len(other_files) > 0:
    print(f"sample of unexpected files: {other_files[:5]}")

num of mysterious files: 0


#delete zone.identifier directly in bash (project folder)

cd raw_data/audio_and_txt_files
find . -name "*:Zone.Identifier" -type f -delete

#agora sim:

In [6]:
import pandas as pd

## Check patient diagnosis

In [20]:
path = '../raw_data/patient_diagnosis.csv'

# load CSV
df = pd.read_csv(path, names=['patient_id', 'diagnosis'])

print("head:")
display(df.head())

print("\ndiagnosis count:")
display(df['diagnosis'].value_counts())

print("\n % of each diagnosis:")
display(df['diagnosis'].value_counts(normalize=True) * 100)

head:


,patient_id,diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI



diagnosis count:


diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Bronchiolitis      6
Pneumonia          6
LRTI               2
Asthma             1
Name: count, dtype: int64


 % of each diagnosis:


diagnosis
COPD              50.793651
Healthy           20.634921
URTI              11.111111
Bronchiectasis     5.555556
Bronchiolitis      4.761905
Pneumonia          4.761905
LRTI               1.587302
Asthma             0.793651
Name: proportion, dtype: float64

the count seems low to me... is there a chance one patiente recorded more than 1 audio?

## Check demograpic info

In [25]:
# Load to see the structure
df_raw = pd.read_csv('../raw_data/demographic_info.txt', sep='\s+', header=None)

print("raw shape:", df_raw.shape)
print("\n 1st 5 rows (raw):")
display(df_raw.head())

raw shape: (126, 6)

 1st 5 rows (raw):


,0,1,2,3,4,5
0,101,3.00,F,NaN,19.0,99.0
1,102,0.75,F,NaN,9.8,73.0
2,103,70.00,F,33.00,NaN,NaN
3,104,70.00,F,28.47,NaN,NaN
4,105,7.00,F,NaN,32.0,135.0


In [26]:
demo_path = '../raw_data/demographic_info.txt'

# Column names - based documentation
demo_cols = ['patient_id', 'age', 'sex', 'bmi_adult', 'weight_child', 'height_child']

# Load the file (it is space-separated, so delim_whitespace=True)
df_demo = pd.read_csv(demo_path, names=demo_cols, sep='\s+')

print("- demographic data head -")
display(df_demo.head())

print("\n- summary stats -")
display(df_demo.describe())

print("\n- gender distribution -")
display(df_demo['sex'].value_counts())

- demographic data head -


,patient_id,age,sex,bmi_adult,weight_child,height_child
0,101,3.00,F,NaN,19.0,99.0
1,102,0.75,F,NaN,9.8,73.0
2,103,70.00,F,33.00,NaN,NaN
3,104,70.00,F,28.47,NaN,NaN
4,105,7.00,F,NaN,32.0,135.0



- summary stats -


,patient_id,age,bmi_adult,weight_child,height_child
count,126.000000,125.00000,75.000000,44.000000,42.000000
mean,163.500000,42.99264,27.190000,21.361136,104.652381
std,36.517119,32.20907,5.372519,17.150885,30.793128
min,101.000000,0.25000,16.500000,7.140000,64.000000
25%,132.250000,4.00000,24.150000,11.755000,81.250000
50%,163.500000,60.00000,27.400000,15.100000,99.500000
75%,194.750000,71.00000,29.185000,24.325000,117.750000
max,226.000000,93.00000,53.500000,80.000000,183.000000



- gender distribution -


sex
M    79
F    46
Name: count, dtype: int64

## Check file name


In [27]:
# Check the naming logic
with open('../raw_data/filename_format.txt', 'r') as f:
    print("--- Filename Format Legend ---")
    print(f.read())

--- Filename Format Legend ---
Elements contained in the filenames:

Patient number (101,102,...,226)
Recording index
Chest location (Trachea (Tc), {Anterior (A), Posterior (P), Lateral (L)}{left (l), right (r)})
Acquisition mode (sequential/single channel (sc), simultaneous/multichannel (mc))
Recording equipment (AKG C417L Microphone, 3M Littmann Classic II SE Stethoscope, 3M Litmmann 3200 Electronic Stethoscope, WelchAllyn Meditron Master Elite Electronic Stethoscope)
